# 🎙️ intiVoice AI — Studio Suara Pintar (Versi Resmi: V1.2.6)

Notebook ini berfungsi sebagai **Engine Backend GPU Serverless** bertenaga Tesla T4 untuk aplikasi **intiVoice Web Studio**.

### ✨ Fitur Utama V1.2.6:
1. 💾 **Persistent Google Drive Cache**: Bobot model disimpan permanen di Drive. Sekali unduh, sesi berikutnya **100% INSTAN** tanpa unduh ulang!
2. 🎙️ **Ultimate Voice Cloning**: Kloning vokal 1:1 via audio continuation + transcript (mereproduksi nafas & nuansa mikro).
3. 🎛️ **Controllable Voice Cloning**: Kloning warna vokal (timbre) dari berkas audio referensi.
4. ⚡ **Zero-Reload VRAM Cache**: Menghentikan (Stop) dan Menjalankan (Run) ulang cell hanya butuh 1 detik.
5. 🔊 **DSP Phase-Vocoder Time-Stretch**: Kontrol kecepatan bicara (0.7x - 1.5x) bebas distorsi nada/chipmunk.
6. 🎬 **Dual Subtitle CapCut Generator**: Otomatis membuat .SRT & .ASS Karaoke (Mobile 9:16 safe zone).
7. 📁 **Fail-Safe Dual-Save**: Otomatis mengarsipkan 4 berkas (.wav, .txt, .srt, .ass) ke Google Drive Anda.

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. Berikan izin koneksi Google Drive saat pop-up muncul agar cache model tersimpan permanen.
3. Tunggu hingga URL Cloudflare Tunnel muncul (format: ).
4. Salin URL tersebut ke aplikasi Web Studio pada **Tab Pengaturan Mesin**.


In [ ]:
"""
================================================================================
🎙️ INTIVOICE AI — ENGINE GPU & CLOUDFLARE TUNNEL (V1.2.6 PERSISTENT CACHE)
================================================================================
🚀 FITUR TERBARU V1.2.6:
   1. 💾 Persistent Google Drive Cache: Model 4.6GB disimpan permanen di Drive.
      (Sekali download, sesi berikutnya 100% INSTAN tanpa download ulang!).
   2. 🎙️ Ultimate Voice Cloning: Kloning vokal 1:1 via audio continuation + transcript.
   3. 🎛️ Controllable Voice Cloning: Kloning timbre audio referensi dengan style guidance.
   4. ⚡ Zero-Reload VRAM Cache: Model tersimpan di GPU (Stop/Run ulang cuma butuh 1 detik!).
   5. 🔊 DSP Phase-Vocoder Time-Stretch: Kontrol kecepatan bicara (0.7x - 1.5x) bebas distorsi.
   6. 🎬 Dual Subtitle CapCut Generator: Auto-export .SRT & .ASS Karaoke (Mobile 9:16 safe zone).
   7. 📁 Fail-Safe Dual-Save: Otomatis arsipkan 4 file (.wav, .txt, .srt, .ass) ke Google Drive.
================================================================================
"""

import os
import sys
import time
import subprocess
import threading
import re
import io
import random
import base64
import tempfile
from typing import Optional, List

# ------------------------------------------------------------------------------
# 1. VERIFIKASI HARDWARE GPU TESLA T4
# ------------------------------------------------------------------------------
print("=" * 80)
print("🚀 MEMULAI INTIVOICE AI — ENGINE GPU & CLOUDFLARE TUNNEL (V1.2.6)")
print("=" * 80)

print("\n[1/6] 🔍 Memeriksa Akselerator Hardware GPU...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
        print(f"   ✅ GPU Terdeteksi: {gpu_name} ({vram_total:.0f} MB VRAM)")
    else:
        print("   ⚠️ PERINGATAN: GPU Tidak Terdeteksi! Model akan berjalan lambat di CPU.")
        print("   👉 Klik Runtime -> Change runtime type -> Pilih T4 GPU lalu Restart!")
except Exception as e:
    print(f"   ⚠️ Gagal memeriksa GPU: {e}")

# ------------------------------------------------------------------------------
# 2. INTEGRASI GOOGLE DRIVE PRIBADI (PERSISTENT MODEL CACHE & AUTO-SAVE)
# ------------------------------------------------------------------------------
# KUNCI INSTAN REFERENSI: Hubungkan Google Drive DI AWAL dan set HF_HOME ke Drive!
# Ini membuat bobot model 4.6GB tersimpan permanen di akun Drive Anda.
# Sesi berikutnya Colab langsung membaca file lokal Drive tanpa download internet lagi!
print("\n[2/6] 📁 Menghubungkan Google Drive untuk Cache Model Permanen...")
DRIVE_MOUNTED = False
DRIVE_OUTPUT = "/content/drive/MyDrive/intiVoice_Audio"
HF_CACHE = "/content/drive/MyDrive/intiVoice_Audio/model_cache"

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("   ⏳ Menghubungkan ke Google Drive pribadi Anda...")
        drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    os.makedirs(HF_CACHE, exist_ok=True)
    
    # Arahkan cache Hugging Face ke Google Drive secara permanen
    os.environ['HF_HOME'] = HF_CACHE
    os.environ['HF_HUB_CACHE'] = HF_CACHE
    print(f"   ✅ Google Drive Terhubung!")
    print(f"   💾 Cache Model Permanen : {HF_CACHE}")
    print(f"   📂 Output Hasil Audio   : {DRIVE_OUTPUT}")
except Exception as e:
    print(f"   ℹ️ Google Drive tidak terhubung ({e}). Menggunakan disk lokal sementara...")
    DRIVE_OUTPUT = "/content/outputs"
    HF_CACHE = "/content/model_cache"
    os.environ['HF_HOME'] = HF_CACHE
    os.environ['HF_HUB_CACHE'] = HF_CACHE
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    os.makedirs(HF_CACHE, exist_ok=True)

# ------------------------------------------------------------------------------
# 3. INSTALASI DEPENDENSI ULTRA-FAST VIA UV
# ------------------------------------------------------------------------------
print("\n[3/6] ⚡ Memeriksa & memasang dependensi (Ultra-Fast via uv)...")
def install_fast_dependencies():
    try:
        import uv
    except ImportError:
        print("   ⏳ Memasang paket manager uv (kecepatan 10x pip)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    
    try:
        import voxcpm
        import soundfile
        import librosa
        import fastapi
        import uvicorn
        import pydantic
        print("   ✅ Seluruh pustaka audio & AI sudah terpasang!")
    except ImportError:
        print("   ⏳ Memasang dependensi inti...")
        subprocess.check_call(["uv", "pip", "install", "--system", "voxcpm", "soundfile", "librosa", "gradio", "fastapi", "uvicorn", "pydantic"])
        print("   ✅ Dependensi berhasil dipasang!")

install_fast_dependencies()

import numpy as np
import soundfile as sf
import librosa
from voxcpm import VoxCPM
from fastapi import FastAPI, Response, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# ------------------------------------------------------------------------------
# 4. MEMASANG CLOUDFLARED (ZERO-CONFIG TUNNEL)
# ------------------------------------------------------------------------------
print("\n[4/6] 🌐 Menyiapkan Cloudflare Tunnel...")
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("   ⏳ Mengunduh binary cloudflared resmi...")
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])
    print("   ✅ Cloudflared siap digunakan!")
else:
    print("   ✅ Cloudflared sudah terpasang.")

# ------------------------------------------------------------------------------
# 5. INISIALISASI MODEL KE GPU (DENGAN CACHE GOOGLE DRIVE & ZERO-RELOAD VRAM)
# ------------------------------------------------------------------------------
print("\n[5/6] 🧠 Memuat Model Suara AI ke Memori GPU...")
if 'model' not in globals():
    t_start = time.time()
    
    # Cek apakah cache model sudah ada di Google Drive
    has_cached_weights = os.path.exists(HF_CACHE) and len(os.listdir(HF_CACHE)) > 0
    if has_cached_weights and DRIVE_MOUNTED:
        print("   ⚡ Model ditemukan di Google Drive! Memuat langsung dari cache Drive (tanpa download internet)...")
    else:
        print("   ⏳ Mengunduh bobot model dari Hugging Face (~4.6GB)...")
        print("   💡 Bobot akan disimpan otomatis ke Google Drive Anda agar run berikutnya 100% instan!")

    model = VoxCPM.from_pretrained(
        'openbmb/VoxCPM2',
        cache_dir=HF_CACHE,
        load_denoiser=False
    )
    
    load_duration = time.time() - t_start
    vram_used = torch.cuda.memory_allocated(0) / (1024 ** 2) if torch.cuda.is_available() else 0
    print(f"   ✅ MODEL SUARA AI SIAP DI GPU ({load_duration:.2f}s) | VRAM Terpakai: {vram_used:.1f} MB")
else:
    vram_used = torch.cuda.memory_allocated(0) / (1024 ** 2) if torch.cuda.is_available() else 0
    print(f"   ⚡ RE-ATTACH BERHASIL: Model sudah aktif di GPU VRAM ({vram_used:.1f} MB) tanpa reload!")

# ------------------------------------------------------------------------------
# 6. SMART AUTO-CHUNKER & FASTAPI BACKEND (ULTIMATE CLONING & AUDIO PRECISION)
# ------------------------------------------------------------------------------
print("\n[6/6] 🚀 Menyiapkan Smart Auto-Chunker & Backend API...")

api_app = FastAPI(title="intiVoice Engine Colab")
api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    prompt_voice: Optional[str] = ""
    reference_audio_b64: Optional[str] = None
    prompt_text: Optional[str] = None
    cfg_value: Optional[float] = 2.0
    temperature: Optional[float] = 2.0
    speed: Optional[float] = 1.0
    seed: Optional[int] = 42

def format_timestamp_srt(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    if millis >= 1000:
        millis = 999
    return f"{hrs:02d}:{mins:02d}:{secs:02d},{millis:03d}"

def format_timestamp_ass(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    centis = int(round((seconds - int(seconds)) * 100))
    if centis >= 100:
        centis = 99
    return f"{hrs:01d}:{mins:02d}:{secs:02d}.{centis:02d}"

def split_text_into_smart_chunks(text: str, max_chars_per_chunk: int = 750) -> List[str]:
    """Smart sentence splitter yang mematuhi batas token aman VRAM."""
    clean_text = text.replace('\r\n', '\n').replace('\r', '\n').strip()
    if len(clean_text) <= max_chars_per_chunk:
        return [clean_text] if clean_text else []

    sentence_pattern = r'(?<=[.!?。！？\n])\s+'
    raw_sentences = [s.strip() for s in re.split(sentence_pattern, clean_text) if s.strip()]

    chunks = []
    current_chunk = ""

    for s in raw_sentences:
        if len(current_chunk) + len(s) + 1 <= max_chars_per_chunk:
            current_chunk = f"{current_chunk} {s}".strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            if len(s) > max_chars_per_chunk:
                clause_pattern = r'(?<=[,;:\-—，、])\s+'
                clauses = [c.strip() for c in re.split(clause_pattern, s) if c.strip()]
                sub_chunk = ""
                for c in clauses:
                    if len(sub_chunk) + len(c) + 1 <= max_chars_per_chunk:
                        sub_chunk = f"{sub_chunk} {c}".strip()
                    else:
                        if sub_chunk:
                            chunks.append(sub_chunk)
                        sub_chunk = c
                if sub_chunk:
                    chunks.append(sub_chunk)
                current_chunk = ""
            else:
                current_chunk = s

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

@api_app.get("/")
@api_app.get("/health")
def root():
    return {
        "status": "online",
        "ready": True,
        "ok": True,
        "engine": "intiVoice AI Engine (Colab T4)",
        "version": "1.2.5",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "cloning_supported": True,
        "cloning_modes": ["controllable", "ultimate"],
        "sample_rate": model.tts_model.sample_rate if 'model' in globals() else 48000,
    }

# ------------------------------------------------------------------------------
# 6. ENGINE SINTESIS SUARA & ASYNC JOB QUEUE (ANTI-TIMEOUT 100s CLOUDFLARE)
# ------------------------------------------------------------------------------
jobs = {}

def _internal_synthesize(req: TTSRequest, progress_fn=None) -> dict:
    if not req.text or not req.text.strip():
        raise HTTPException(status_code=400, detail="Teks naskah tidak boleh kosong!")
    
    temp_files = []
    try:
        total_len = len(req.text)
        
        # 1. Kunci seed konstan secara absolut (DNA suara identik untuk seluruh kalimat)
        target_seed = int(req.seed) if (req.seed is not None and req.seed >= 0) else 42
        torch.manual_seed(target_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(target_seed)
        np.random.seed(target_seed)
        random.seed(target_seed)

        # 2. Pastikan CFG guidance scale aktif (default resmi: 2.0, min: 1.0)
        target_cfg = 2.0
        if req.cfg_value is not None and float(req.cfg_value) >= 1.0:
            target_cfg = float(req.cfg_value)
        elif req.temperature is not None and float(req.temperature) >= 1.0:
            target_cfg = float(req.temperature)

        # 3. Ekstrak tag emosi di awal naskah jika ada (misal: (whispering softly...))
        emotion_match = re.match(r'^\s*(\([^)]+\))\s*', req.text)
        global_emotion = emotion_match.group(1).strip() if emotion_match else ""

        # Format prefix kontrol karakter & emosi
        style_parts = []
        if req.prompt_voice:
            clean_voice = re.sub(r"[()（）]", "", req.prompt_voice).strip()
            if clean_voice:
                style_parts.append(clean_voice)
        if global_emotion:
            clean_emo = re.sub(r"[()（）]", "", global_emotion).strip()
            if clean_emo:
                style_parts.append(clean_emo)
        
        clean_control = ", ".join(style_parts)

        # Bersihkan tag emosi di awal naskah teks
        base_text = re.sub(r'^\s*\([^)]+\)\s*', '', req.text).strip()
        if not base_text:
            base_text = req.text.strip()

        # 4. Tangani Berkas Audio Referensi (Voice Cloning: Controllable & Ultimate Mode)
        ref_wav_path = None
        prompt_text_clean = None
        is_ultimate_cloning = False

        if req.reference_audio_b64 and req.reference_audio_b64.strip():
            try:
                audio_bytes = base64.b64decode(req.reference_audio_b64)
                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_in:
                    tmp_in.write(audio_bytes)
                    raw_audio_path = tmp_in.name
                    temp_files.append(raw_audio_path)

                # Pre-processing Audio Referensi: Resample ke 16kHz Mono (Native encoder AudioVAE)
                y, sr = librosa.load(raw_audio_path, sr=16000, mono=True)
                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_16k:
                    sf.write(tmp_16k.name, y, 16000)
                    ref_wav_path = tmp_16k.name
                    temp_files.append(ref_wav_path)

                # Cek apakah mode Ultimate Cloning aktif (Audio + Transkrip Kata Persis)
                if req.prompt_text and req.prompt_text.strip():
                    prompt_text_clean = req.prompt_text.strip()
                    is_ultimate_cloning = True
                    clean_control = ""
                    print(f"   🎙️ [Ultimate Voice Cloning] Aktif! Meniru 100% nuansa vokal mikro ({len(prompt_text_clean)} karakter transkrip)")
                else:
                    print(f"   🎛️ [Controllable Voice Cloning] Aktif! Meniru timbre vokal dasar pembicara")
            except Exception as clone_err:
                print(f"   ⚠️ Gagal memproses audio kloning: {clone_err}. Melanjutkan dengan mode Voice Design biasa.")
                ref_wav_path = None
                prompt_text_clean = None

        # 5. Smart Chunking: Ambang batas aman 750 karakter
        chunks = split_text_into_smart_chunks(base_text, max_chars_per_chunk=750)
        num_chunks = len(chunks)
        
        mode_desc = "Ultimate Clone" if is_ultimate_cloning else ("Timbre Clone" if ref_wav_path else "Voice Design")
        print("-" * 80)
        print(f"🎮 [GPU T4 Processing] Mode: {mode_desc} | Naskah: {total_len} karakter | {num_chunks} Bagian | Seed: #{target_seed} | CFG: {target_cfg}")
        if clean_control:
            print(f"   🎛️ Control Instruction: '({clean_control})'")
        if req.speed and abs(float(req.speed) - 1.0) > 0.02:
            print(f"   ⚡ Kecepatan Bicara: {float(req.speed):.2f}x (DSP Phase-Vocoder Time-Stretch)")
        
        sample_rate = model.tts_model.sample_rate
        silence_pause = np.zeros(int(sample_rate * 0.25), dtype=np.float32)
        audio_segments = []
        chunk_durations = []

        if progress_fn:
            progress_fn(f"Menyiapkan {num_chunks} bagian naskah di GPU...")

        for i, chunk in enumerate(chunks):
            clean_chunk = chunk.replace('\n', ' ').strip()
            clean_chunk = re.sub(r'\s+', ' ', clean_chunk)
            if not clean_chunk:
                continue

            t_chunk = time.time()
            prompt = f"({clean_control}){clean_chunk}" if clean_control else clean_chunk

            gen_kwargs = {
                "text": prompt,
                "cfg_value": target_cfg,
                "inference_timesteps": 10,
            }

            if ref_wav_path:
                gen_kwargs["reference_wav_path"] = ref_wav_path
                if is_ultimate_cloning and prompt_text_clean:
                    gen_kwargs["prompt_wav_path"] = ref_wav_path
                    gen_kwargs["prompt_text"] = prompt_text_clean

            with torch.inference_mode():
                wav = model.generate(**gen_kwargs)
            
            audio_segments.append(wav)
            chunk_durations.append(len(wav) / sample_rate)
            if i < num_chunks - 1:
                audio_segments.append(silence_pause)
            
            dur_chunk = time.time() - t_chunk
            print(f"   ✅ Bagian [{i+1}/{num_chunks}] selesai dalam {dur_chunk:.2f}s")
            if progress_fn:
                progress_fn(f"Bagian [{i+1}/{num_chunks}] selesai ({dur_chunk:.1f}s)")

        full_audio = np.concatenate(audio_segments) if len(audio_segments) > 1 else audio_segments[0]
        
        # 6. TERAPKAN KONTROL KECEPATAN BICARA (LIBROSA TIME-STRETCH)
        target_speed = float(req.speed) if req.speed is not None else 1.0
        if abs(target_speed - 1.0) > 0.03:
            if progress_fn:
                progress_fn("Menyesuaikan kecepatan tempo bicara (DSP)...")
            t_stretch = time.time()
            full_audio = librosa.effects.time_stretch(full_audio, rate=target_speed)
            print(f"   ⚡ Kecepatan bicara disesuaikan ke {target_speed:.2f}x dalam {time.time() - t_stretch:.2f}s")

        final_duration = len(full_audio) / sample_rate
        print(f"🎉 SUKSES: Durasi Akhir {final_duration:.2f}s | Sample Rate: {sample_rate}Hz")

        # 7. SIMPAN OTOMATIS 4-BERKAS KE GOOGLE DRIVE
        random_id = f"{random.randint(1000, 9999)}"
        timestamp_str = time.strftime('%Y%m%d_%H%M%S')
        base_name = f"audio_{timestamp_str}_{random_id}"
        
        save_wav_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.wav")
        save_txt_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.txt")
        save_srt_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.srt")
        save_ass_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.ass")

        try:
            sf.write(save_wav_path, full_audio, sample_rate, format='WAV')
            with open(save_txt_path, 'w', encoding='utf-8') as f:
                f.write(f"=== METADATA SINTESIS SUARA INTIVOICE AI ===\n")
                f.write(f"Waktu       : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Karakter    : {req.prompt_voice or '-'}\n")
                f.write(f"Mode        : {mode_desc}\n")
                f.write(f"Seed DNA    : {target_seed}\n")
                f.write(f"CFG Scale   : {target_cfg}\n")
                f.write(f"Speed       : {target_speed}x\n")
                f.write(f"Durasi Audio: {final_duration:.2f} detik\n")
                f.write(f"Total Karakter: {total_len}\n")
                f.write(f"Jumlah Bagian : {num_chunks}\n\n")
                f.write(f"--- NASKAH TEKS ---\n{base_text}\n")
            
            scale_ratio = 1.0 / target_speed if target_speed > 0 else 1.0
            srt_lines = []
            curr_time = 0.0
            pause_sec = 0.25 * scale_ratio

            for idx, (c_text, dur) in enumerate(zip(chunks, chunk_durations)):
                seg_dur = dur * scale_ratio
                start_str = format_timestamp_srt(curr_time)
                end_str = format_timestamp_srt(curr_time + seg_dur)
                srt_lines.append(f"{idx + 1}\n{start_str} --> {end_str}\n{c_text.strip()}\n")
                curr_time += seg_dur + pause_sec
            
            with open(save_srt_path, 'w', encoding='utf-8') as f:
                f.write("\n".join(srt_lines) + "\n")

            ass_header = """[Script Info]
Title: intiVoice Studio Audio Subtitles
ScriptType: v4.00+
WrapStyle: 0
ScaledBorderAndShadow: yes
YCbCr Matrix: None
PlayResX: 1080
PlayResY: 1920

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Montserrat,58,&H00FFFFFF,&H0000D2B4,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,3.5,1.5,2,70,70,280,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
            ass_events = []
            curr_time_ass = 0.0
            for c_text, dur in zip(chunks, chunk_durations):
                seg_dur = dur * scale_ratio
                start_ass = format_timestamp_ass(curr_time_ass)
                end_ass = format_timestamp_ass(curr_time_ass + seg_dur)
                clean_cap = c_text.strip().replace('\n', ' ')
                ass_events.append(f"Dialogue: 0,{start_ass},{end_ass},Default,,0,0,0,,{clean_cap}")
                curr_time_ass += seg_dur + pause_sec

            with open(save_ass_path, 'w', encoding='utf-8') as f:
                f.write(ass_header + "\n".join(ass_events) + "\n")

            print(f"   💾 Auto-Save 4-Berkas Berhasil:")
            print(f"      🎵 WAV : {save_wav_path}")
            print(f"      📄 TXT : {save_txt_path}")
            print(f"      🎬 SRT : {save_srt_path} (CapCut / Premiere Ready)")
            print(f"      ✨ ASS : {save_ass_path} (TikTok / Shorts Safe Zone 9:16)")
        except Exception as drive_err:
            print(f"   ⚠️ Gagal menyimpan arsip ke Drive: {drive_err}")

        out_buf = io.BytesIO()
        sf.write(out_buf, full_audio, sample_rate, format='WAV')
        out_buf.seek(0)
        wav_bytes = out_buf.read()
        b64_str = f"data:audio/wav;base64,{base64.b64encode(wav_bytes).decode('utf-8')}"

        return {
            "wav_bytes": wav_bytes,
            "data_url": b64_str,
            "final_duration": final_duration,
            "mode_desc": mode_desc,
            "base_name": base_name,
        }

    except Exception as e:
        print(f"❌ ERROR SINTESIS: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        for f in temp_files:
            try:
                if os.path.exists(f):
                    os.unlink(f)
            except:
                pass

@api_app.post("/job/create")
def create_job(req: TTSRequest):
    if not req.text or not req.text.strip():
        raise HTTPException(status_code=400, detail="Teks naskah tidak boleh kosong!")
    
    job_id = f"job_{int(time.time()*1000)}_{random.randint(100, 999)}"
    jobs[job_id] = {
        "status": "processing",
        "progress": "Menyiapkan naskah di GPU...",
        "created_at": time.time(),
        "result": None,
        "error": None
    }
    
    def worker():
        try:
            def on_prog(msg):
                if job_id in jobs:
                    jobs[job_id]["progress"] = msg
            res = _internal_synthesize(req, progress_fn=on_prog)
            if job_id in jobs:
                jobs[job_id]["status"] = "completed"
                jobs[job_id]["progress"] = "Selesai!"
                jobs[job_id]["result"] = {
                    "audio_b64": res["data_url"],
                    "duration": res["final_duration"],
                    "mode": res["mode_desc"],
                    "filename": f"{res['base_name']}.wav"
                }
        except Exception as err:
            if job_id in jobs:
                jobs[job_id]["status"] = "failed"
                jobs[job_id]["error"] = str(err)
                print(f"❌ Error Async Job {job_id}: {err}")
    
    worker_t = threading.Thread(target=worker, daemon=True)
    worker_t.start()
    return {"job_id": job_id, "status": "processing"}

@api_app.get("/job/status/{job_id}")
def get_job_status(job_id: str):
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job ID tidak ditemukan atau sudah kedaluwarsa")
    item = jobs[job_id]
    resp = {
        "job_id": job_id,
        "status": item["status"],
        "progress": item.get("progress", ""),
        "error": item.get("error")
    }
    if item["status"] == "completed" and item.get("result"):
        resp["result"] = item["result"]
    return resp

@api_app.post("/generate")
def generate_tts(req: TTSRequest):
    res = _internal_synthesize(req)
    return Response(
        content=res["wav_bytes"],
        media_type="audio/wav",
        headers={
            "Content-Disposition": f"attachment; filename={res['base_name']}.wav",
            "X-Audio-Duration": f"{res['final_duration']:.2f}",
            "X-Audio-Cloning-Mode": res["mode_desc"],
        }
    )

# # 7. MENJALANKAN SERVER FASTAPI & CLOUDFLARE TUNNEL (NON-BLOCKING LOG)
# ------------------------------------------------------------------------------
def run_fastapi():
    import uvicorn
    uvicorn.run(api_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_fastapi, daemon=True)
server_thread.start()
time.sleep(2)

print("\n" + "=" * 80)
print("🌐 MEMBUKA CLOUDFLARE TUNNEL KE INTERNET PUBLIK...")
print("=" * 80)

tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_tunnel_wait = time.time()

while time.time() - start_tunnel_wait < 30:
    line = tunnel_process.stdout.readline()
    if not line:
        continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "🎉" * 40)
    print("🚀 MESIN INTIVOICE AI SUDAH ONLINE DAN SIAP DIGUNAKAN!")
    print("=" * 80)
    print(f"👉 SALIN URL INI KE WEB STUDIO (Tab Pengaturan Mesin):")
    print(f"   {tunnel_url}")
    print("=" * 80)
    print("💡 CATATAN:")
    print("   • Buka Web Studio intiVoice di browser Anda.")
    print("   • Buka Tab Pengaturan Mesin -> Tempel URL di atas -> Klik 'Simpan & Uji Ping'.")
    print("   • Status akan berubah menjadi 'Terhubung ke Engine GPU (Colab T4)'.")
    print("   • Dukungan Kloning Vokal (Controllable & Ultimate Mode) aktif sepenuhnya!")
    print("🎉" * 40 + "\n")
else:
    print("\n❌ Gagal mendapatkan URL Cloudflare. Periksa koneksi internet Google Colab.")

# Menjaga proses tetap aktif dengan silent loop
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Menghentikan server engine...")
